In [3]:
folder_aiff = '/Users/yerik/_apple_lib/_a_progs/_a2ms_env/_9_ML_project/data/raw/_audio_files/04_03_house_spkr_soundcloud_audio'

In [4]:
# ----------------------------######----------------------------#
# _cv_1001_overlaypng_SAFE_GET_aiff                             #
# ----------------------------######----------------------------#

import os
import uuid
from io import BytesIO
from tqdm import tqdm

from PIL import Image
from mutagen.aiff import AIFF
from mutagen.id3 import ID3, APIC, ID3NoHeaderError


def _cv_1001_overlaypng_SAFE_GET_aiff(
    folder_aiff,
    overlay_png_path,
    recurse=True,

    quadrant=2,
    overlay_scale=0.65,
    anchor="center",
    offset_xy=(0, 0),
    max_overlay_px=None,
    blend_alpha=1.0,

    skip_if_no_cover=True,
    blank_size=(1400, 1400),
):

    # -------------------------
    # FILE ITERATOR
    # -------------------------
    def _iter_aiff(root):
        exts = (".aiff", ".aif")
        if recurse:
            for r, _, files in os.walk(root):
                for fn in files:
                    if fn.lower().endswith(exts) and not fn.startswith(".") and not fn.startswith("._"):
                        yield os.path.join(r, fn)
        else:
            for fn in os.listdir(root):
                p = os.path.join(root, fn)
                if os.path.isfile(p) and fn.lower().endswith(exts):
                    yield p

    # -------------------------
    # GEOMETRY
    # -------------------------
    def _quadrant_rect(w, h, q):
        row = (q - 1) // 3
        col = (q - 1) % 3
        x0 = int(col * (w / 3))
        y0 = int(row * (h / 3))
        x1 = int((col + 1) * (w / 3))
        y1 = int((row + 1) * (h / 3))
        return x0, y0, x1, y1

    def _anchor_point(x0, y0, x1, y1, which):
        cx = (x0 + x1) // 2
        cy = (y0 + y1) // 2
        mapping = {
            "center": (cx, cy),
            "nw": (x0, y0),
            "n": (cx, y0),
            "ne": (x1, y0),
            "w": (x0, cy),
            "e": (x1, cy),
            "sw": (x0, y1),
            "s": (cx, y1),
            "se": (x1, y1),
        }
        return mapping.get(which.lower(), (cx, cy))

    def _apply_alpha(img, mult):
        if mult >= 0.999:
            return img
        r, g, b, a = img.split()
        a = a.point(lambda v: int(v * mult))
        return Image.merge("RGBA", (r, g, b, a))

    # -------------------------
    # LOAD OVERLAY
    # -------------------------
    overlay_base = Image.open(overlay_png_path).convert("RGBA")

    results = []
    paths = list(_iter_aiff(folder_aiff))

    for path in tqdm(paths, desc="AIFF COVER OVERLAY"):

        tmp = os.path.join(
            os.path.dirname(path),
            f".__tmp__{uuid.uuid4().hex}__{os.path.basename(path)}"
        )

        try:
            # -------------------------
            # BIT-PERFECT COPY
            # -------------------------
            with open(path, "rb") as fsrc, open(tmp, "wb") as fdst:
                fdst.write(fsrc.read())

            # -------------------------
            # LOAD AIFF
            # -------------------------
            audio = AIFF(tmp)

            if audio.tags is None:
                audio.tags = ID3()

            tags = audio.tags

            # -------------------------
            # GET COVER
            # -------------------------
            apics = tags.getall("APIC")
            apic = apics[0] if apics else None

            if apic:
                base = Image.open(BytesIO(apic.data)).convert("RGBA")
                mime = apic.mime
                desc = apic.desc
                apic_type = apic.type
            else:
                if skip_if_no_cover:
                    os.remove(tmp)
                    results.append({"Path": path, "status": "skip_no_cover"})
                    continue

                base = Image.new("RGBA", blank_size, (0, 0, 0, 255))
                mime = "image/png"
                desc = "Cover"
                apic_type = 3

            # -------------------------
            # OVERLAY (EXACT SYSTEM)
            # -------------------------
            W, H = base.size
            x0, y0, x1, y1 = _quadrant_rect(W, H, quadrant)
            qW, qH = (x1 - x0), (y1 - y0)

            ov = _apply_alpha(overlay_base.copy(), blend_alpha)

            target_w = int(qW * overlay_scale)
            target_h = int(qH * overlay_scale)

            if max_overlay_px:
                target_w = min(target_w, max_overlay_px)
                target_h = min(target_h, max_overlay_px)

            ov.thumbnail((target_w, target_h))

            ow, oh = ov.size

            ax, ay = _anchor_point(x0, y0, x1, y1, anchor)
            dx, dy = offset_xy
            ax += dx
            ay += dy

            if anchor in ("center", "n", "s"):
                px = ax - ow // 2
            elif anchor in ("ne", "e", "se"):
                px = ax - ow
            else:
                px = ax

            if anchor in ("center", "w", "e"):
                py = ay - oh // 2
            elif anchor in ("sw", "s", "se"):
                py = ay - oh
            else:
                py = ay

            base.paste(ov, (int(px), int(py)), ov)

            # -------------------------
            # REPLACE ONLY COVER
            # -------------------------
            tags.delall("APIC")

            out = BytesIO()
            base.save(out, format="PNG")

            tags.add(APIC(
                encoding=3,
                mime="image/png",
                type=apic_type,
                desc=desc,
                data=out.getvalue()
            ))

            # -------------------------
            # SAVE + REPLACE
            # -------------------------
            audio.save()
            os.replace(tmp, path)

            results.append({"Path": path, "status": "ok"})

        except Exception as e:
            results.append({"Path": path, "status": "error", "detail": str(e)})
            if os.path.exists(tmp):
                os.remove(tmp)

    return results

In [5]:
res = _cv_1001_overlaypng_SAFE_GET_aiff(
    folder_aiff=folder_aiff,
    overlay_png_path="/Users/yerik/Music/_2_MP3_group_RK/img_MP3_2025.png",
    quadrant=2,
    overlay_scale=0.65,
    anchor="center",
    offset_xy=(0,0),
)

AIFF COVER OVERLAY: 100%|█████████████████████████████████████████████████████| 21/21 [00:03<00:00,  5.33it/s]
